# Response generation

[Fine-tuning Llama 2 with LoRA](https://deci.ai/blog/fine-tune-llama-2-with-lora-for-question-answering/)

# Fine-tuning
new_model will be the name of your fine-tuned model (saved)

In [1]:
import os, torch, logging
from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, HfArgumentParser, TrainingArguments, pipeline
from peft import LoraConfig, PeftModel
from trl import SFTTrainer
from huggingface_hub import login
from flash_attn import flash_attn_func
import accelerate
import wandb
from tqdm import tqdm


In [2]:
%load_ext dotenv
%dotenv

login(token=os.environ.get("HF_TOKEN", ""), add_to_git_credential=True)
wandb.login(key=os.environ.get("WANDB_API_KEY", ""), relogin=True)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.


Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (cache).
Your token has been saved to /home/user/.cache/huggingface/token
Login successful


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /home/user/.netrc


True

In [3]:
# Initialize Wandb
wandb_config = {
    "base_model": "llama-2-chat-7b",
}
wandb.init(
    job_type="fine-tuning",
    config=wandb_config,
    project="emotion-chat-bot-ncu",
    group="candidate_generation",
    mode="online",
    resume="auto"
)

wandb: Currently logged in as: yangyx30678. Use `wandb login --relogin` to force relogin


In [4]:
# Dataset
training_data = load_dataset("json", data_files="dataset_v1.json", split="train")
# Model and tokenizer names
base_model_name = "meta-llama/Llama-2-7b-chat-hf"
refined_model = "llama-2-7b-testing"

# Tokenizer
llama_tokenizer = AutoTokenizer.from_pretrained(base_model_name, trust_remote_code=True)
llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "right"  # Fix for fp16

# Quantization Config
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)
wandb.config["quantization_configuration"] = quant_config.to_dict() if quant_config is not None else {}

# Model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    attn_implementation="flash_attention_2",
    quantization_config=quant_config,
    device_map={"": 0},
    use_cache=False,
    low_cpu_mem_usage=True
)

base_model.config.use_cache = False
base_model.config.pretraining_tp = 1

Generating train split: 0 examples [00:00, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [5]:
# LoRA Config
peft_parameters = LoraConfig(
    lora_alpha=16,
    lora_dropout=0.1,
    r=8,
    bias="none",
    task_type="CAUSAL_LM"
)
wandb.config["lora_configuration"] = peft_parameters.to_dict()

# Training Params
train_params = TrainingArguments(
    output_dir="./results_modified",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    fp16=False,
    bf16=False,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    group_by_length=True,
    lr_scheduler_type="constant",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},
    report_to=["wandb"],
    overwrite_output_dir=True
    # torch_compile=True
)
wandb.config["trainer_arguments"] = train_params.to_dict()

# Trainer
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=training_data,
    peft_config=peft_parameters,
    
    dataset_text_field='formatted_chat', #also changed
    
    tokenizer=llama_tokenizer,
    args=train_params
)

# Training
fine_tuning.train()
wandb.finish()
# Save Model
fine_tuning.model.save_pretrained(refined_model)

/home/user/.cache/pypoetry/virtualenvs/chat-bot-20tW9agt-py3.11/lib/python3.11/site-packages/trl/trainer/sft_trainer.py:225: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(


Map:   0%|          | 0/700 [00:00<?, ? examples/s]

/home/user/.cache/pypoetry/virtualenvs/chat-bot-20tW9agt-py3.11/lib/python3.11/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
The input hidden states seems to be silently casted in float32, this might be related to the fact you have upcasted embedding or layer norm layers in float32. We will cast back the input in torch.float16.


Step,Training Loss
25,2.653500
50,1.261700
75,1.146900
100,1.156100
125,1.042100
150,1.166600
175,0.956400
200,1.129500
225,0.936400
250,1.023700


Checkpoint destination directory ./results_modified/checkpoint-25 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results_modified/checkpoint-50 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results_modified/checkpoint-75 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results_modified/checkpoint-100 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results_modified/checkpoint-125 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results_modified/checkpoint-150 already exists and is non-empty. Saving will proceed but saved results may be invalid.
Checkpoint destination directory ./results_modified/checkpoint-175 already exis

train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇████
train/grad_norm,▄▄▂▁▃▁▃▂▂▃▁▃▂▂▃▃▄▂▃▃▄▃▅▃▄▄▄▇▅▆▆█▄▆▃
train/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/loss,█▃▃▃▃▃▂▃▂▃▃▂▃▂▃▂▂▂▂▃▂▂▂▂▂▂▂▁▂▁▁▂▁▂▁
train/total_flos,▁
train/train_loss,▁
train/train_runtime,▁
train/train_samples_per_second,▁
train/train_steps_per_second,▁
train/epoch,5.0


# Inference

In [6]:
test_dataset = load_dataset("json", data_files="test_dataset_v1.json", split="train")

In [7]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=False
)

# Model
inference_model = AutoModelForCausalLM.from_pretrained(
    refined_model,
    attn_implementation="flash_attention_2",
    quantization_config=quant_config,
    device_map={"": 0},
    use_cache=False,
    low_cpu_mem_usage=True
)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [8]:
pipe = pipeline(task="text-generation", model=inference_model, tokenizer=llama_tokenizer)
OGpipe = pipeline(task="text-generation", model=base_model, tokenizer=llama_tokenizer)

generated_texts = []
OGgenerated_texts = []

pbar = tqdm(total=len(test_dataset))
for example in test_dataset:
    chat_text = example['formatted_chat']
    generated_text = pipe(chat_text, max_new_tokens=200)[0]['generated_text']
    OGgenerated_text = OGpipe(chat_text, max_new_tokens=200)[0]['generated_text']
    
    generated_texts.append(generated_text)
    OGgenerated_texts.append(OGgenerated_text)
    pbar.update(1)
    
pbar.close()


  0%|          | 0/20 [00:00<?, ?it/s]`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
/home/user/.cache/pypoetry/virtualenvs/chat-bot-20tW9agt-py3.11/lib/python3.11/site-packages/torch/utils/checkpoint.py:90: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
 50%|█████     | 10/20 [05:04<05:00, 30.10s/it]/home/user/.cache/pypoetry/virtualenvs/chat-bot-20tW9agt-py3.11/lib/python3.11/site-packages/transformers/pipelines/base.py:1157: UserWarning: You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
  warnings.warn(
100%|██████████| 20/20 [10:12<00:00, 30.64s/it]


In [18]:
for i in range(10):
    print(i+1 , test_dataset[i]['chat'][1]['content'], ":\n")
    print("finetuned:\n", generated_texts[i].split("[/INST]",1)[1])
    print('- ' * 75)
    print("OG:\n", OGgenerated_texts[i].split("[/INST]",1)[1])
    print("OG:\n", OGgenerated_texts[i].split("[/INST]",1)[1])
    print('=' * 150, "\n\n")

1  Yes , that's true . Damn ! I just lost half my paper , and now I can't even do my homework.This is a bad time for this to happen .  :

finetuned:
  Yes, you've got that right! Can you believe I just lost half of my paper?! Ugh, this is absolutely infuriating! Now, not only do I have to deal with my homework, but I've got to go back and redo everything! What a terrible time for this to happen! 😤🤬🔥😞 

Seriously, who thought it was a good idea to have a paper disappear like that?! I've been working on it for hours! And now, because of some silly mistake, I've got to start all over again. What a waste of time! 😡😞🤯

This is just ridiculous! Why does this keep happening to me?! Can't anything ever go right for once?! 😡😞🤷‍♀️

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
OG:
  Absolutely infuriating! I just lost half my paper, and now I can't even do my homework. This is absolutely the 